<a href="https://colab.research.google.com/github/pleasewaitinthequeue/pythonic-investing-with-alpaca/blob/main/Alpaca_Trading_API_Shared.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Alpaca Paper API Library
Using the alpaca.trading.client library, and the google colab secrets library, we can set up a trading client.  

The Trading client can allow us to check out our current positions, and send trading orders.

##THIS IS NOT INVESTING ADVICE, NOR IS IT DIRECTION TO BUY SPECIFIC STOCKS

**Let's try that out!**

In [ ]:
!pip install alpaca-py

## Secured with oAuth
User's Client ID / Secret are specific to the trading account.  There can be a 1:Many relationship between User and Account.  Before you start, create your alpaca trading account, and security store your API KEY / SECRET in Colab User Variables, or in an Environment Variable.

**DO NOT PUT YOUR API KEY / SECRET IN YOUR CODE**

In [ ]:
from alpaca.trading.client import TradingClient
from google.colab import userdata
api_key = userdata.get('ALPACA_API_KEY')
secret_key = userdata.get('ALPACA_API_SECRET')

client = TradingClient(api_key, secret_key, paper=True)

The account has a number of properties:

In [ ]:
import pprint

account = client.get_account()
print(f"Equity: {account.equity}")
print(f"Buying Power: {account.buying_power}")

# Pretty Print will help us easily list all the properties of an object such as "Trade Account"
pprint.pp(account)

## Open Positions
** You Own A Share of `AAPL` ** and You Want to See it!

In [ ]:
# Get all active positions
positions = client.get_all_positions()

# Check if you hold a specific symbol
try:
  position = client.get_open_position("AAPL")
except BaseException as e:
  print(f'Oops, something bad happened. {e}')

### Submit an Order
As simple as parametrizing an object, and sending that object with the client.  

In [ ]:
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

# BUYS A SHARE OF ROCKET LABS
market_order_data = MarketOrderRequest(
    symbol="RKLB",
    qty=1,
    side=OrderSide.BUY,
    time_in_force=TimeInForce.DAY
)

market_order = client.submit_order(order_data=market_order_data)

### Get Open Positions
list all positions for a specific account.

In [ ]:
# Get all active positions
positions = client.get_all_positions()
print(f'{positions}')

# Pretty Print will help us easily list all the properties of an object such as "Trade Account"
pprint.pp(account)

Let's get some historical stock data

In [ ]:
from alpaca.data.historical.stock import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime

# HISTORICAL CLIENT CAN RETRIEVE TRENDING INFORMATION
data_client = StockHistoricalDataClient(api_key, secret_key)

request_params = StockBarsRequest(
    symbol_or_symbols="RKLB",
    timeframe=TimeFrame.Day,
    start=datetime(2024, 1, 1)
)

bars = data_client.get_stock_bars(request_params)
# Convert to a pandas dataframe
df = bars.df

# By Default Historical Bars have a Multi Index, we will drop it to make working with the data frame easier.
print(f'{df.head()}')
print(f'{df.index.droplevel(0)}')

# Set Index to single index with timestamp
df.index = df.index.droplevel(0)

#Show New Index
print(f'{df.index}')
#Show First Five Rows of New Data Frame.
print(f'{df.head()}')

### MACD Crossover

Technical Analysis Technique - we'll come back to this another time.
https://commodity.com/technical-analysis/macd/

The important part here is that Technical Analysis can help you create and interpret **BUY**/**SELL** signals which you would then act on.

In [ ]:
!pip install matplotlib

In [ ]:
# # Calculate MACD values using the pandas_ta library
#df.ta.macd(close='close', fast=12, slow=26, signal=9, append=True)
# Get the 26-day EMA of the closing price
k = df['close'].ewm(span=12, adjust=False, min_periods=12).mean()
# Get the 12-day EMA of the closing price
d = df['close'].ewm(span=26, adjust=False, min_periods=26).mean()
# Subtract the 26-day EMA from the 12-Day EMA to get the MACD
macd = k - d
# Get the 9-Day EMA of the MACD for the Trigger line
macd_s = macd.ewm(span=9, adjust=False, min_periods=9).mean()
# Calculate the difference between the MACD - Trigger for the Convergence/Divergence value
macd_h = macd - macd_s
# Add all of our new values for the MACD to the dataframe
df['macd'] = df.index.map(macd)
df['macd_h'] = df.index.map(macd_h)
df['macd_s'] = df.index.map(macd_s)
# View our data
pd.set_option("display.max_columns", None)
print(df)

In [ ]:
import plotly.subplots as subplots
import plotly.graph_objects as go
import numpy as np

# Construct a 2 x 1 Plotly figure
fig = subplots.make_subplots(rows=2, cols=1)

# price Line
fig.append_trace(
    go.Scatter(
        x=df.index,
        y=df['open'],
        line=dict(color='#ff9900', width=1),
        name='open',
        # showlegend=False,
        legendgroup='1',

    ), row=1, col=1
)

# Candlestick chart for pricing
fig.append_trace(
    go.Candlestick(
        x=df.index,
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        increasing_line_color='#ff9900',
        decreasing_line_color='black',
        showlegend=False

    ), row=1, col=1
)

# Fast Signal (%k)
fig.append_trace(
    go.Scatter(
        x=df.index,
        y=df['macd_h'],
        line=dict(color='#ff9900', width=2),
        name='macd',
        # showlegend=False,
        legendgroup='2',

    ), row=2, col=1
)

# Slow signal (%d)
fig.append_trace(
    go.Scatter(
        x=df.index,
        y=df['macd_s'],
        line=dict(color='#000000', width=2),
        # showlegend=False,
        legendgroup='2',
        name='signal'
    ), row=2, col=1
)

# Make it pretty
layout = go.Layout(
    plot_bgcolor='#efefef',
    # Font Families
    font_family='Monospace',
    font_color='#000000',
    font_size=20,
    xaxis=dict(
        rangeslider=dict(
            visible=False
        )
    )
)

# Update options and show plot
fig.update_layout(layout)
fig.show()

In [ ]:
import matplotlib.pyplot as plt

#MACD Line is just SHORT MOVING AVERAGE - LONG MOVING AVERAGE
def calculate_macd(prices, short_window=12, long_window=26, signal_window=9):
   short_ema = prices.ewm(span=short_window, adjust=False).mean()
   long_ema = prices.ewm(span=long_window, adjust=False).mean()
   macd_line = short_ema - long_ema
   signal_line = macd_line.ewm(span=signal_window, adjust=False).mean()
   return macd_line, signal_line

macd, signal = calculate_macd(df['close'])
df['macd'] = macd
df['signal'] = signal

plt.figure(figsize=(12, 6))
# Plot stock prices
plt.plot(df['close'], label='Stock Prices', color='blue')
# Plot MACD and Signal lines
plt.plot(df['macd'] * df['close'], label='MACD Line', color='red')
plt.plot(df['signal'] * df['close'], label='Signal Line', color='green')
plt.title('Stock Prices and MACD Indicator')
plt.xlabel('Days')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()

## A simple Script to Get some Stock Price History

In [ ]:
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime, timedelta

# Define request parameters
request_params = StockBarsRequest(
    symbol_or_symbols="SPY", # Gets State Street SPDR S&P 500 ETF Trust (SPY)
    timeframe=TimeFrame.Day, # Days Represent Rows
    start=datetime.now() - timedelta(days=120), #Starting 120 Days Ago
    end=datetime.now() - timedelta(days=1) # Till Yesterday
)

# Fetch bars and convert to a pandas DataFrame
bars = data_client.get_stock_bars(request_params)
df = bars.df

In [ ]:
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

def submit_market_order(symbol, qty, side):
    """
    Submit a market order through Alpaca's Trading API.
    """
    order_data = MarketOrderRequest(
        symbol=symbol,
        qty=qty,
        side=side,
        time_in_force=TimeInForce.DAY
    )
    order = client.submit_order(order_data=order_data)
    return order

# LETS DO SOME TRADE!!!!!

### Summary of the "Sin Baskets"
An agent (Gemini 2.5 Flash) designed this trading strategy to profit off of people's investment in and indulence in sin.  This is not trading advice!  I just wanted to have fun with it to see how the profile does.
*   **Pride**: `RACE`, `AAPL`, `LULU` (Status & Brand)
*   **Envy**: `META`, `SNAP`, `PINS` (Social Comparison)
*   **Wrath**: `LMT`, `RTX`, `GD` (Defense & Conflict)
*   **Gluttony**: `MCD`, `PEP`, `HSY` (Indulgence)
*   **Lust**: `MTCH`, `BMBL`, `RICK` (Desire & Nightlife)
*   **Sloth**: `NFLX`, `DASH`, `UBER` (Convenience & Escapism)
*   **Greed**: `GS`, `JPM`, `LVS` (Finance & Gambling)

In [ ]:
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.trading.client import TradingClient
from google.colab import userdata

# TODO:  Set up API KEY / SECRET as ENV Variable
api_key = userdata.get('ALPACA_API_KEY')
secret_key = userdata.get('ALPACA_API_SECRET')

# Initialize Trading Client
client = TradingClient(api_key, secret_key, paper=True)

# Divide Buying Power by Buckets, and Buckets by Tickers
bucket_buying_power = float( account.buying_power ) / 7.0 / 3.0
# Get Today's Date
todays_datetime = datetime.today().strftime('%Y-%m-%d')
print(f'{todays_datetime} we have per bucket buying power of:  {bucket_buying_power}')

sinbuckets = {
    "Strategy":
    [{
        'Sin':'Pride',
        'Tickers':['RACE','AAPL','LULU']
    },{
        'Sin':'Envy',
        'Tickers':['META','SNAP','PINS']
    },{
        'Sin':'Wrath',
        'Tickers':['LMT','RTX','GD']
    },{
        'Sin':'Gluttony',
        'Tickers':['MCD','PEP','HSY']
    },{
        'Sin':'Lust',
        'Tickers':['MTCH','BMBL','RICK']
    },{
        'Sin':'Sloth',
        'Tickers':['NFLX','DASH','UBER']
    },{
        'Sin':'Greed',
        'Tickers':['GS','JPM','LVS']
    }]
}

# For Each Bucket
for sin in sinbuckets['Strategy']:
  # For Each Ticker in Bucket
  for tick in sin['Tickers']:
    #what is ticker price (Today at Close)?
    request_params = StockBarsRequest(symbol_or_symbols=tick,timeframe=TimeFrame.Day,start=todays_datetime)
    df = data_client.get_stock_bars(request_params).df
    # Get Rid of Multi Index, so that we can get most recent Close.
    df.index = df.index.droplevel(0)
    # Print to Console
    print(f'{df.iloc[0]['close']}')
    price_at_close = df.iloc[0]['close']
    print(f'Symbol:  {tick} closed at:  {df.iloc[0]['close']}')
    # // Integer Divide Buying Power by Price at Close.
    num_shares = bucket_buying_power // price_at_close
    # If > 0 then Buy Some Shares!
    if num_shares > 0:
      order_data = MarketOrderRequest(
          symbol=tick,
          qty=num_shares,
          side=OrderSide.BUY,
          time_in_force=TimeInForce.DAY
      )
      # We are Submitting Market Orders, so the Price Might be Different than when we submit.
      market_order = client.submit_order(order_data=order_data)
      print(f'Sin {sin['Sin']}:  Submitted {tick} order for {num_shares:.0f} shares at {price_at_close:,.2f} for ~ Market Buy of $ {num_shares * price_at_close:,.2f}.')
    else:
      print(f'Sin {sin['Sin']}:  No shares to buy for {tick}.')


##Alternative Energy Trading Strategy
Gemini Flash 2.5 also generated this plan.  It made some quantitative comparisons and selection related to whether or not the stock was a good pick.

###Solar Energy
* FSLR (First Solar) - NASDAQ, S&P 500. Large-cap solar panel manufacturer.
* ENPH (Enphase Energy) - NASDAQ, S&P 500. Microinverters and battery storage.
* SEDG (SolarEdge Technologies) - NASDAQ. Inverters and smart energy management.
* RUN (Sunrun) - NASDAQ. Residential solar panel installations.
* CSIQ (Canadian Solar) - NASDAQ. Global solar panel manufacturer.
###Wind Energy
* NEE (NextEra Energy) - S&P 500. World's largest generator of renewable energy from wind and sun.
* GE (General Electric) - S&P 500. Massive player in wind turbine manufacturing.
* XEL (Xcel Energy) - NASDAQ, S&P 500. Major utility and one of the largest wind energy providers in the US.
* AES (The AES Corporation) - S&P 500. Global power company with significant wind development.
* TPIC (TPI Composites) - NASDAQ. Manufacturer of composite wind blades.
###Hydroelectric Power
* PCG (PG&E Corporation) - S&P 500. Operates one of the largest investor-owned hydroelectric systems in the US.
* DUK (Duke Energy) - S&P 500. Major utility managing numerous hydroelectric plants.
* PEG (Public Service Enterprise Group) - S&P 500. Diversified utility with hydro and pumped storage assets.
* EIX (Edison International) - S&P 500. Major player in renewable hydroelectric generation in California.
* ED (Consolidated Edison) - S&P 500. Utility heavily investing in clean energy and transmission for hydro.
###Hydrogen & Fuel Cells
* PLUG (Plug Power) - NASDAQ. Turnkey hydrogen fuel cell solutions.
* FCEL (FuelCell Energy) - NASDAQ. Fuel cell power plants for stationary power generation.
* BLDP (Ballard Power Systems) - NASDAQ. PEM fuel cell products for heavy-duty motive applications.
* APD (Air Products and Chemicals) - S&P 500. World's largest supplier of hydrogen and investing heavily in green hydrogen.
* CMI (Cummins Inc.) - S&P 500. Major manufacturer investing significantly in hydrogen electrolyzers and fuel cells.
Geothermal Energy
* CVX (Chevron) - Dow Jones, S&P 500. Making heavy investments in emerging closed-loop geothermal technologies.
* BRK.B (Berkshire Hathaway) - S&P 500. BHE Renewables is a massive player in geothermal via the Imperial Valley projects.
* BKR (Baker Hughes) - NASDAQ, S&P 500. Oilfield services giant actively expanding its geothermal energy technology division.
* SLB (Schlumberger) - S&P 500. Pivoting heavily into geothermal via its New Energy division.
* OXY (Occidental Petroleum) - S&P 500. Significant investments in geothermal and direct air capture via Oxy Low Carbon Ventures.
###Biomass & Biofuels
* VLO (Valero Energy) - S&P 500. One of the largest producers of renewable diesel.
* GPRE (Green Plains) - NASDAQ. Leading biorefining company focused on ethanol and sustainable ingredients.
* CLNE (Clean Energy Fuels) - NASDAQ. Provider of renewable natural gas (RNG) for transportation.
* AMTX (Aemetis) - NASDAQ. Renewable fuels and biochemicals company.
* ALTO (Alto Ingredients) - NASDAQ. Producer of specialty alcohols and essential ingredients.
###Nuclear Energy (Low-Carbon Baseload)
*CEG (Constellation Energy) - NASDAQ, S&P 500. The largest producer of carbon-free nuclear energy in the US.
* SO (Southern Company) - S&P 500. Major utility that recently completed the Vogtle nuclear expansion.
* AEP (American Electric Power) - NASDAQ, S&P 500. Significant utility with nuclear baseload generation.
* ETR (Entergy) - S&P 500. Operates a large fleet of nuclear power plants.
* D (Dominion Energy) - S&P 500. Major utility with strong nuclear and offshore wind portfolios.

In [ ]:
from zoneinfo import ZoneInfo
# why o why is it may 1st already? -- Answer:  datetime returns UTC Date Time, but we happened to be in EST
print(f'{datetime.today().strftime('%Y-%m-%d')}')
# It was after 8pm when the computer gave tomorrow's date, instead of today's date.
print(f'{datetime.now(tz=ZoneInfo("EST")).strftime('%Y-%m-%d')}')

In [ ]:
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.trading.client import TradingClient
from google.colab import userdata

# API KEY and SECRET are different for Each Account.
api_key = userdata.get('ALT_NRG_API_KEY')
secret_key = userdata.get('ALT_NRG_API_SECRET')

# Trading Client and Data Client are Different Sessions.
client = TradingClient(api_key, secret_key, paper=True)
account = client.get_account()
data_client = StockHistoricalDataClient(api_key, secret_key)

altnrg_buckets = {
    "Strategy":
    [{
        'Type':'Solar Energy',
        'Tickers':['FSLR','ENPH','SEDG','RUN','CSIQ']
    },{
        'Type':'Wind Energy',
        'Tickers':['NEE','GE','XEL','AES','BEPC'] #,'TPIC'] -- TPIC was on the NASDAQ or NYSE or S&P - it was on something called OTC
    },
    {
        'Type':'Hydroelectric Power',
        'Tickers':['PCG','DUK','PEG','EIX','ED']
    },{
        'Type':'Hydrogen & Fuel Cells',
        'Tickers':['PLUG','FCEL','BLDP','APD','CMI']
    },{
        'Type':'Geothermal Energy',
        'Tickers':['CVX','BRK.B','BKR','SLB','OXY']
    },{
        'Type':'Biomass & Biofuels',
        'Tickers':['VLO','GPRE','CLNE','AMTX','ALTO']
    },{
        'Type':'Nuclear Energy (Low-Carbon Baseload)',
        'Tickers':['CEG','SO','AEP','ETR','D']
    }]
}

# Get Bucket Buying Power, but Instead of constant, divide by number of Buckets.
bucket_buying_power = float( account.buying_power ) / len( altnrg_buckets['Strategy'] ) / 5.0
todays_datetime = datetime.now(tz=ZoneInfo("EST")).strftime('%Y-%m-%d') #converted to EST
print(f'{todays_datetime} we have per bucket buying power of:  {bucket_buying_power}')

df = None

for nrg in altnrg_buckets['Strategy']:
  for tick in nrg['Tickers']:
    #what is ticker price?
    request_params = StockBarsRequest(symbol_or_symbols=tick,timeframe=TimeFrame.Day,start=todays_datetime)
    df = data_client.get_stock_bars(request_params).df
    df.index = df.index.droplevel(0)
    price_at_close = df.iloc[0]['close']
    print(f'Symbol:  {tick} closed at:  {df.iloc[0]['close']}')
    num_shares = bucket_buying_power // price_at_close
    if num_shares > 0:
      order_data = MarketOrderRequest(
          symbol=tick,
          qty=num_shares,
          side=OrderSide.BUY,
          time_in_force=TimeInForce.DAY
      )
      market_order = client.submit_order(order_data=order_data)
      print(f'Type {nrg['Type']}:  Submitted {tick} order for {num_shares:.0f} shares at {price_at_close:,.2f} for ~ Market Buy of $ {num_shares * price_at_close:,.2f}.')
    else:
      print(f'Sin {ngr['Type']}:  No shares to buy for {tick}.')

In [ ]:
from alpaca.trading.client import TradingClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime
from google.colab import userdata
api_key = userdata.get('ALPACA_API_KEY')
secret_key = userdata.get('ALPACA_API_SECRET')

data_client = TradingClient(api_key, secret_key)

print(f'{data_client}')

In [ ]:
# This Code Does not Work Correctly.
from alpaca.data.requests import CorporateActionsRequest
from datetime import datetime

def get_corporate_actions(symbol, start_date=None, end_date=None, limit=100):
    """
    Retrieves corporate actions data for a given symbol from Alpaca's Data API.

    Args:
        symbol (str): The stock symbol (e.g., "AAPL").
        start_date (str, optional): Start date in 'YYYY-MM-DD' format. Defaults to None.
        end_date (str, optional): End date in 'YYYY-MM-DD' format. Defaults to None.
        limit (int, optional): The maximum number of corporate actions to retrieve. Defaults to 100.

    Returns:
        list: A list of CorporateAction objects if successful, None otherwise.
    """
    # data_client is expected to be initialized globally from a previous cell
    # (e.g., data_client = StockHistoricalDataClient(api_key, secret_key)).

    # Convert date strings to datetime objects for the request
    start_dt = datetime.strptime(start_date, '%Y-%m-%d') if start_date else None
    end_dt = datetime.strptime(end_date, '%Y-%m-%d') if end_date else None

    request_params = CorporateActionsRequest(
        symbol=symbol,
        start=start_dt,
        end=end_dt,
        limit=limit
    )

    try:
        corporate_actions = data_client.get_corporate_actions(request_params)
        return corporate_actions
    except Exception as e:
        print(f"Error retrieving corporate actions for {symbol}: {e}")
        return None


In [ ]:
get_corporate_actions('AAPL','2026-01-01','2026-06-01')

In [ ]:
# This Code Looks Like It Works, but then I discovered that Alpaca Didn't have corporate actions for most tickers.
from alpaca.data.historical.corporate_actions import CorporateActionsClient
from alpaca.data.requests import CorporateActionsRequest

# Initialize client with your API keys
api_key = userdata.get('ALPACA_API_KEY')
secret_key = userdata.get('ALPACA_API_SECRET')
client = CorporateActionsClient(api_key, secret_key)

# Request specific actions within a date range
request = CorporateActionsRequest(
    types=["cash_dividend", "stock_dividend"],
    start="2026-01-01",
    end="2026-05-14"
)

# Fetch and convert to DataFrame
data = client.get_corporate_actions(request).df

sinbuckets = {
    "Strategy":
    [{
        'Sin':'Pride',
        'Tickers':['RACE','AAPL','LULU'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Envy',
        'Tickers':['META','SNAP','PINS'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Wrath',
        'Tickers':['LMT','RTX','GD'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Gluttony',
        'Tickers':['MCD','PEP','HSY'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Lust',
        'Tickers':['MTCH','BMBL','RICK'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Sloth',
        'Tickers':['NFLX','DASH','UBER'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Greed',
        'Tickers':['GS','JPM','LVS'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    }]
}

ticker_data = data[data['symbol'] == 'AAPL'].sort_values(by='ex_date', ascending=False)
print(ticker_data)

ticker_data = data[data['symbol'] == 'LMT'].sort_values(by='ex_date', ascending=False)
print(ticker_data)

for sin in sinbuckets['Strategy']:
  for tick in sin['Tickers']:
    ticker_data = data[data['symbol'] == tick].sort_values(by='ex_date', ascending=False)
    print(ticker_data)
    if not ticker_data.empty:
      sin['Yield'][sin['Tickers'].index(tick)] = f'${ticker_data.iloc[0]['rate']:,.2f}'
      sin['YieldDate'][sin['Tickers'].index(tick)] = ticker_data.iloc[0]['ex_date'].strftime('%Y-%m-%d')

print(sinbuckets)


In [ ]:
!pip install yfinance

In [ ]:
# Yahoo Finance is More Reliable for Historical Dividend Information
import yfinance as yf
import pandas as pd

ticker = 'LMT'

tick = yf.Ticker( ticker )

#print(f'{tick} {tick.dividends.tail(1)} {tick.info} {type(tick.dividends.tail(1))}')

print(f'{tick.dividends.tail(1)[0]}')



In [ ]:
import yfinance as yf

sinbuckets = {
    "Strategy":
    [{
        'Sin':'Pride',
        'Tickers':['RACE','AAPL','LULU'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Envy',
        'Tickers':['META','SNAP','PINS'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Wrath',
        'Tickers':['LMT','RTX','GD'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Gluttony',
        'Tickers':['MCD','PEP','HSY'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Lust',
        'Tickers':['MTCH','BMBL','RICK'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Sloth',
        'Tickers':['NFLX','DASH','UBER'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    },{
        'Sin':'Greed',
        'Tickers':['GS','JPM','LVS'],
        'Yield':[0,0,0],
        'YieldDate':[None,None,None]
    }]
}


for sin in sinbuckets['Strategy']:
  for tick in sin['Tickers']:
    ticker_data = yf.Ticker( tick )
    ticker_div = ticker_data.dividends.tail(1)
    if not ticker_div.empty:
      sin['Yield'][sin['Tickers'].index(tick)] = f'${ticker_div.iloc[0]:,.2f}'
      sin['YieldDate'][sin['Tickers'].index(tick)] = ticker_div.index[0].strftime('%Y-%m-%d')

print(f'{sinbuckets}')

In [ ]:
import pprint

pprint.pprint(sinbuckets)

In [ ]:
import yfinance as yf
import pprint

altnrg_buckets = {
    "Strategy":
    [{
        'Type':'Solar Energy',
        'Tickers':['FSLR','ENPH','SEDG','RUN','CSIQ'],
        'Yield':[0,0,0,0,0],
        'YieldDate':[None,None,None,None,None]
    },{
        'Type':'Wind Energy',
        'Tickers':['NEE','GE','XEL','AES','BEPC'], #,'TPIC'] -- TPIC was on the NASDAQ or NYSE or S&P - it was on something called OTC
        'Yield':[0,0,0,0,0],
        'YieldDate':[None,None,None,None,None]
    },
    {
        'Type':'Hydroelectric Power',
        'Tickers':['PCG','DUK','PEG','EIX','ED'],
        'Yield':[0,0,0,0,0],
        'YieldDate':[None,None,None,None,None]
    },{
        'Type':'Hydrogen & Fuel Cells',
        'Tickers':['PLUG','FCEL','BLDP','APD','CMI'],
        'Yield':[0,0,0,0,0],
        'YieldDate':[None,None,None,None,None]
    },{
        'Type':'Geothermal Energy',
        'Tickers':['CVX','BRK.B','BKR','SLB','OXY'],
        'Yield':[0,0,0,0,0],
        'YieldDate':[None,None,None,None,None]
    },{
        'Type':'Biomass & Biofuels',
        'Tickers':['VLO','GPRE','CLNE','AMTX','ALTO'],
        'Yield':[0,0,0,0,0],
        'YieldDate':[None,None,None,None,None]
    },{
        'Type':'Nuclear Energy (Low-Carbon Baseload)',
        'Tickers':['CEG','SO','AEP','ETR','D'],
        'Yield':[0,0,0,0,0],
        'YieldDate':[None,None,None,None,None]
    }]
}

for nrg in altnrg_buckets['Strategy']:
  for tick in nrg['Tickers']:
    ticker_data = yf.Ticker( tick )
    ticker_div = ticker_data.dividends.tail(1)
    if not ticker_div.empty:
      nrg['Yield'][nrg['Tickers'].index(tick)] = f'${ticker_div.iloc[0]:,.2f}'
      nrg['YieldDate'][nrg['Tickers'].index(tick)] = ticker_div.index[0].strftime('%Y-%m-%d')

pprint.pprint(altnrg_buckets)